In [1]:
# Growfinix Internship - Task 2

# Real Estate RAG System
## Intelligent Hotel Property Search Using LangChain, Hugging Face and ChromaDB

### Project Overview

# This project implements a Retrieval-Augmented Generation (RAG) system
# for intelligent hotel property search.

# The system uses a large hotel dataset containing hotel names,
# descriptions, facilities, locations, ratings and nearby attractions.

# Hotel information is converted into vector embeddings and stored
# in ChromaDB. When a user submits a natural-language query, the system
# retrieves semantically relevant hotel properties and uses a Large
# Language Model to generate a grounded response.

### Example Query

# "Show me modern hotels with large balconies, swimming pools
# and beautiful views."

In [2]:
import os
import re
import warnings
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
DATA_PATH = "../data/hotels.csv"

df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
    encoding="latin1"  # Added this line to handle non-utf-8 characters
)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Dataset loaded successfully!
Rows: 1010033
Columns: 16


,countyCode,countyName,cityCode,cityName,HotelCode,HotelName,HotelRating,Address,Attractions,Description,FaxNumber,HotelFacilities,Map,PhoneNumber,PinCode,HotelWebsiteUrl
0,AL,Albania,106078,Albanien,1003300,De Paris Hotel,FourStar,Nr. 7 Brigada Viii Street Tirane,NaN,Hotel de Paris is a charming boutique hotel th...,42268822,Private parking Parking onsite Television in c...,41.32213|19.81665,00355 4226 5009,1000,https://www.booking.com/hotel/al/de-paris.html
1,AL,Albania,106078,Albanien,1003301,Hotel Green,FourStar,Rruga Kavajes. Kombinat Km 2. Vaqarr VaqarrTir...,NaN,"Located in a suburb of Tirana, Hotel Green is ...",35548520058,airport pick up wifi available in all areas Ai...,41.30413|19.74703,+35548520057,1041,https://www.booking.com/hotel/al/hotel-green.html
2,AL,Albania,106078,Albanien,1003302,Theranda Hotel,ThreeStar,Rr. Andon Zako Cajupi Villa 6 & 7 Villa 6 & 7T...,NaN,"Set in Tirana, 1.2 km from Skanderbeg Square, ...",00355 (0)42273689,face masks for guests available all plates cu...,41.3216|19.81199,00355 (0)42273766,1019,https://www.booking.com/hotel/al/theranda.html
3,AL,Albania,106078,Albanien,1003303,Seven Hotel,ThreeStar,"KAVAJA STREET, CLOSE TURKISH AMBASSY TIRANA",Skanderbeg Square: within 500 metre,This hotel enjoys an enviable setting in Tiran...,NaN,À la carte dinner Breakfast buffet Breakfast C...,41.328027|19.815052,NaN,1001,http://www.hotelseven-tirana.com/
4,AL,Albania,106078,Albanien,1003325,Viktoria,ThreeStar,Rruga E Elbasanit Km 4 Sauk SaukTirana,NaN,Located in a new residential area at the edge ...,+355695406986,internet services Ironing service Family rooms...,41.29125|19.85349,355 69 5406986,1000,https://www.booking.com/hotel/al/viktoria-sauk...


In [4]:
print("Dataset Shape:", df.shape)

Dataset Shape: (1010033, 16)


In [5]:
print(df.columns.tolist())

['countyCode', ' countyName', ' cityCode', ' cityName', ' HotelCode', ' HotelName', ' HotelRating', ' Address', ' Attractions', ' Description', ' FaxNumber', ' HotelFacilities', ' Map', ' PhoneNumber', ' PinCode', ' HotelWebsiteUrl']


In [6]:
missing_values = df.isnull().sum()

display(
    missing_values.sort_values(
        ascending=False
    )
)

 FaxNumber          560347
 Attractions        525092
 PhoneNumber        327137
 HotelWebsiteUrl    250118
 HotelFacilities     50378
 Description         47005
 PinCode             30979
 Map                   930
countyCode             912
 Address               102
 HotelRating             0
 HotelName               0
 countyName              0
 cityCode                0
 HotelCode               0
 cityName                0
dtype: int64

In [7]:
print(
    "Duplicate rows:",
    df.duplicated().sum()
)

Duplicate rows: 0


In [8]:
important_columns = [
    "hotel_name",
    "Description",
    "HotelFacilities",
    "Attractions",
    "cityName",
    "countyName",
    "HotelRating"
]

available_columns = [
    col for col in important_columns
    if col in df.columns
]

print(
    "Available important columns:"
)

print(available_columns)

Available important columns:
[]


In [9]:
def clean_text(text):
    
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Remove HTML tags
    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )
    
    # Remove excessive whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    
    return text.strip()

In [10]:
# Strip spaces from all column names
df.columns = df.columns.str.strip()

df["clean_description"] = (
    df["Description"]
    .apply(clean_text)
)

In [11]:
df["clean_facilities"] = (
    df["HotelFacilities"]
    .apply(clean_text)
)

In [12]:
df["clean_attractions"] = (
    df["Attractions"]
    .apply(clean_text)
)

In [13]:
df = df[
    df["clean_description"].str.len() > 20
].copy()

print(
    "Hotels with usable descriptions:",
    len(df)
)

Hotels with usable descriptions: 962961


In [14]:
if "HotelCode" in df.columns:
    
    df = df.drop_duplicates(
        subset=["HotelCode"]
    )
else:
    
    df = df.drop_duplicates(
        subset=["hotel_name", "Address"]
    )

df = df.reset_index(drop=True)

print(
    "Hotels after duplicate removal:",
    len(df)
)

Hotels after duplicate removal: 735989


In [15]:
def create_property_document(row):

    hotel_name = str(
        row.get("HotelName", "")
    )

    description = str(
        row.get("clean_description", "")
    )

    facilities = str(
        row.get("clean_facilities", "")
    )

    attractions = str(
        row.get("clean_attractions", "")
    )

    address = str(
        row.get("Address", "")
    )

    city = str(
        row.get("cityName", "")
    )

    country = str(
        row.get("countyName", "")
    )

    rating = str(
        row.get("HotelRating", "")
    )

    hotel_code = str(
        row.get("HotelCode", "")
    )

    return f"""
Hotel Code:
{hotel_code}

Hotel Name:
{hotel_name}

Location:
{city}, {country}

Address:
{address}

Rating:
{rating}

Description:
{description}

Facilities:
{facilities}

Nearby Attractions:
{attractions}
""".strip()

In [16]:
df["property_text"] = df.apply(
    create_property_document,
    axis=1
)

In [17]:
print(
    df["property_text"].iloc[0]
)

Hotel Code:
1003300

Hotel Name:
De Paris Hotel

Location:
Albanien, Albania

Address:
Nr. 7 Brigada Viii Street Tirane 

Rating:
FourStar

Description:
Hotel de Paris is a charming boutique hotel that offers stylish rooms and a large courtyard garden perfect for enjoying a relaxing meal outdoors. A private bar with an open fireplace is suitable for a night cap. Rooms and suites at De Paris are luxuriously furnished and offer air-conditioning, a minibar and a work desk. Every room has a flat-screen TV with satellite channels. Private bathroom provides a hairdryer, a bathrobe and free toiletries. The hotel is situated in the elegant area of the city full of bars and restaurants, and just a few steps away from the main institutions such as Parliament, Palace of Congress, Opera House, Art Gallery and Main Boulevard. Ironing, laundry and dry cleaning services are available. The reception is open 24 hours a day and breakfast buffet is served each day. The nearest bus stop is only 50 metres 

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [19]:
test_text = """
Modern luxury hotel with spacious balconies,
beautiful city views and swimming pool.
"""

test_embedding = embedding_model.embed_query(
    test_text
)

print(
    "Embedding dimensions:",
    len(test_embedding)
)

print(
    "First 10 values:",
    test_embedding[:10]
)

Embedding dimensions: 384
First 10 values: [0.07298368215560913, 0.048572637140750885, -0.011704646982252598, 0.09866354614496231, -0.016576407477259636, 0.02137986198067665, 0.05167635902762413, -0.029505599290132523, -0.05050986632704735, -0.024346403777599335]


In [20]:
from langchain_core.documents import Document

In [21]:
documents = []

for idx, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Creating documents"
):
    
    metadata = {
        "hotel_code": str(
            row.get("HotelCode", idx)
        ),
        "hotel_name": str(
            row.get("hotel_name", "")
        ),
        "city": str(
            row.get("cityName", "")
        ),
        "country": str(
            row.get("countyName", "")
        ),
        "rating": str(
            row.get("HotelRating", "")
        )
    }
    
    documents.append(
        Document(
            page_content=row["property_text"],
            metadata=metadata
        )
    )

Creating documents:   0%|          | 0/735989 [00:00<?, ?it/s]

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [23]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

split_documents = text_splitter.split_documents(
    documents
)

print(
    "Original documents:",
    len(documents)
)

print(
    "After chunking:",
    len(split_documents)
)

Original documents: 735989
After chunking: 3734375


In [24]:
from langchain_chroma import Chroma

In [25]:
CHROMA_PATH = "../chroma_db"

vectorstore = Chroma(
    collection_name="hotel_rag_collection",
    embedding_function=embedding_model,
    persist_directory=CHROMA_PATH
)

print("ChromaDB initialized.")

ChromaDB initialized.


In [26]:
TEST_SIZE = 5000

test_documents = split_documents[:TEST_SIZE]

In [27]:
print(
    "Testing with:",
    len(test_documents),
    "documents"
)

Testing with: 5000 documents


In [28]:
BATCH_SIZE = 250

for start in tqdm(
    range(
        0,
        len(test_documents),
        BATCH_SIZE
    ),
    desc="Indexing test documents"
):
    
    batch = test_documents[
        start:start + BATCH_SIZE
    ]
    
    vectorstore.add_documents(
        batch
    )

print("Test indexing completed.")

Indexing test documents:   0%|          | 0/20 [00:00<?, ?it/s]

Test indexing completed.


In [29]:
query = """
Show me modern hotels with large balconies,
beautiful views and swimming pools.
"""

In [30]:
results = vectorstore.similarity_search(
    query,
    k=5
)

In [31]:
for i, doc in enumerate(
    results,
    start=1
):
    
    print("=" * 80)
    print("RESULT", i)
    print("=" * 80)
    
    print(
        "Hotel:",
        doc.metadata.get(
            "hotel_name"
        )
    )
    
    print(
        "Location:",
        doc.metadata.get(
            "city"
        )
    )
    
    print(
        doc.page_content[:1000]
    )

RESULT 1
Hotel: 
Location: Dhërmi
Nearby Attractions:
RESULT 2
Hotel: 
Location: Gjirokastra
Nearby Attractions:
RESULT 3
Hotel: 
Location: Gjirokastra
Nearby Attractions:
RESULT 4
Hotel: 
Location: Korca
Nearby Attractions:
RESULT 5
Hotel: 
Location: Ksamil
Nearby Attractions:


In [32]:
%pip install -qU langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [33]:
from dotenv import load_dotenv
import os

load_dotenv("../.env", override=True)

if os.getenv("GROQ_API_KEY"):
    print("Groq API key loaded successfully.")
else:
    print("Groq API key not found.")

Groq API key loaded successfully.


In [34]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0,
    max_tokens=600
)
print("LLM initialized.")

LLM initialized.


In [35]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
models = client.models.list()
for m in models.data:
    print(m.id)

openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
qwen/qwen3.6-27b
allam-2-7b
groq/compound
openai/gpt-oss-120b
groq/compound-mini
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi


In [36]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
You are an intelligent hotel recommendation assistant.

Use ONLY the information provided in the retrieved hotel
context to answer the user's question.

Do not invent hotel facilities, prices, ratings, locations,
amenities or attractions.

If information is unavailable, say that it is not available
in the retrieved data.

User Query:
{question}

Retrieved Hotel Information:
{context}

Give a concise and useful response.

For every recommended hotel provide:
- Hotel name
- Location
- Rating if available
- Why it matches the user's request
- Relevant facilities
- Relevant attractions if available
"""
)

In [37]:
def format_context(docs):
    
    context = []
    
    for i, doc in enumerate(
        docs,
        start=1
    ):
        
        context.append(
            f"""
PROPERTY {i}

{doc.page_content}
"""
        )
    
    return "\n".join(context)

In [38]:
def rag_query(
    question,
    k=5
):
    
    # Retrieve
    docs = vectorstore.similarity_search(
        question,
        k=k
    )
    
    # Context
    context = format_context(
        docs
    )
    
    # Prompt
    messages = prompt.format_messages(
        question=question,
        context=context
    )
    
    # Generate
    response = llm.invoke(
        messages
    )
    
    return {
        "answer": response.content,
        "documents": docs
    }

In [39]:
query = """
Show me modern hotels with large balconies,
beautiful views, swimming pools and facilities
suitable for families.
"""

result = rag_query(
    query,
    k=5
)

print(
    result["answer"]
)


<think>
Here's a thinking process:

1.  **Analyze User Query:**
   - **Style:** Modern hotels
   - **Features:** Large balconies, beautiful views, swimming pools
   - **Target Audience:** Facilities suitable for families
   - **Request:** Show me hotels matching these criteria.

2.  **Analyze Retrieved Context:**
   - The context provided is extremely sparse. It only lists:
     - PROPERTY 1: Nearby Attractions: [empty]
     - PROPERTY 2: Nearby Attractions: [empty]
     - PROPERTY 3: Nearby Attractions: [empty]
     - PROPERTY 4: Nearby Attractions: [empty]
     - PROPERTY 5: Nearby Attractions: [empty]
   - There is absolutely no information about hotel names, locations, ratings, facilities, balconies, views, pools, or family suitability.

3.  **Apply Constraints:**
   - "Use ONLY the information provided in the retrieved hotel context to answer the user's question."
   - "Do not invent hotel facilities, prices, ratings, locations, amenities or attractions."
   - "If information is 

In [40]:
print("\nRETRIEVED SOURCES")
print("=" * 80)

for i, doc in enumerate(
    result["documents"],
    start=1
):
    
    print(
        f"{i}. "
        f"{doc.metadata.get('hotel_name')} "
        f"- "
        f"{doc.metadata.get('city')}"
    )


RETRIEVED SOURCES
1.  - Dhërmi
2.  - Gjirokastra
3.  - Gjirokastra
4.  - Korca
5.  - Ksamil


In [41]:
while True:
    
    query = input(
        "\nEnter your hotel requirement "
        "(type 'exit' to stop): "
    )
    
    if query.lower().strip() == "exit":
        break
    
    if not query.strip():
        print(
            "Please enter a query."
        )
        continue
    
    try:
        
        result = rag_query(
            query,
            k=5
        )
        
        print(
            "\n" + "=" * 80
        )
        
        print(
            "AI HOTEL RECOMMENDATION"
        )
        
        print(
            "=" * 80
        )
        
        print(
            result["answer"]
        )
        
        print(
            "\nRETRIEVED HOTELS"
        )
        
        for i, doc in enumerate(
            result["documents"],
            start=1
        ):
            
            print(
                f"{i}. "
                f"{doc.metadata.get('hotel_name')} "
                f"- "
                f"{doc.metadata.get('city')}"
            )
            
    except Exception as e:
        
        print(
            "Error:",
            str(e)
        )


Enter your hotel requirement (type 'exit' to stop):  Give me the name of the hotel having balcony and a king size bed with well furnished room and has some beautiful views. It should also have a swimming pool



AI HOTEL RECOMMENDATION

<think>
Here's a thinking process:

1.  **Analyze User Query:**
   - Requirements: Balcony, King size bed, Well-furnished room, Beautiful views, Swimming pool.
   - Output format: Concise, useful. For each recommended hotel: Name, Location, Rating, Why it matches, Relevant facilities, Relevant attractions.

2.  **Analyze Retrieved Context:**
   - PROPERTY 1: Name: "Lovely house with garden", Location: "Albanien, Albania", Address: "Rruga E Shqipeve 204 3 Floor", Rating: "All"
   - PROPERTY 2: Name: (empty), Location: "Pogradec, Albania", Rating: "All"
   - PROPERTY 3: Name: (empty), Location: "Kavaja, Albania", Rating: "All"
   - PROPERTY 4: Name: (empty), Location: "Albanien, Albania", Rating: "All"
   - PROPERTY 5: Name: (empty), Location: "Albanien, Albania", Rating: "All"
   - *Crucial Observation:* The retrieved context contains almost no details about facilities, room types, views, or amenities. It only lists names (mostly empty), locations, one address,


Enter your hotel requirement (type 'exit' to stop):  Find luxury hotels with spa facilities.



AI HOTEL RECOMMENDATION

<think>
Here's a thinking process:

1.  **Analyze User Query:**
   - Request: "Find luxury hotels with spa facilities."
   - Key criteria: Luxury hotels, spa facilities.

2.  **Analyze Retrieved Context:**
   - PROPERTY 1: Facilities: [Empty]
   - PROPERTY 2: Facilities: [Empty]
   - PROPERTY 3: Facilities: [Empty]
   - PROPERTY 4: Facilities: [Empty]
   - PROPERTY 5: Facilities: [Empty]
   - All properties have empty "Facilities" fields. No names, locations, ratings, or other details are provided in the retrieved context.

3.  **Apply Constraints:**
   - "Use ONLY the information provided in the retrieved hotel context to answer the user's question."
   - "Do not invent hotel facilities, prices, ratings, locations, amenities or attractions."
   - "If information is unavailable, say that it is not available in the retrieved data."
   - Must provide specific fields for recommended hotels if any match.

4.  **Evaluate Match:**
   - The retrieved data contains no


Enter your hotel requirement (type 'exit' to stop):  I need a family-friendly hotel with a swimming pool.



AI HOTEL RECOMMENDATION

<think>
Here's a thinking process:

1.  **Analyze User Query:**
   - Needs: Family-friendly hotel
   - Needs: Swimming pool
   - Implicit: Wants recommendations based on provided data

2.  **Analyze Retrieved Context:**
   - There are 5 properties listed (PROPERTY 1 to PROPERTY 5).
   - All 5 properties have *identical* facility lists:
     - Swimming pool, Live music/performance, Private parking, Beachfront, kids' club, outdoor pool (all year), Private beach area, Air conditioning, Free WiFi, Hot tub/jacuzzi, Free parking, Family rooms, Non-smoking rooms, Terrace, Garden, 24-hour front desk, Bar, Restaurant, Parking.
   - Nearby Attractions: Empty for all.
   - Hotel names, locations, ratings: Not provided in the context. Only "PROPERTY 1", "PROPERTY 2", etc. are given.

3.  **Check Constraints:**
   - Use ONLY provided information.
   - Do not invent facilities, prices, ratings, locations, amenities, or attractions.
   - If info unavailable, state it's not a


Enter your hotel requirement (type 'exit' to stop):  exit
